In [ ]:
!pip install google-adk

In [ ]:
import os
import requests
from typing import Dict, Optional
from google.adk.agents import Agent
from google.adk.agents import SequentialAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.models.lite_llm import LiteLlm
from vertexai.preview.reasoning_engines import AdkApp
import asyncio
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.models.llm_response import LlmResponse
from google.genai import types # For creating response content
from typing import Optional
import logging
from typing import Optional
import sys
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.models.llm_response import LlmResponse
from google.adk.agents import Agent
from google.genai.types import Content, Part
from vertexai.preview.reasoning_engines import AdkApp
from google.adk.tools import google_search

In [ ]:
AGENT_MODEL = "gemini-2.5-flash"
API_KEY = ""
llm_model = LiteLlm(model=AGENT_MODEL, api_key=API_KEY)

In [ ]:
search_agent = Agent(
    name="search_agent",
    model=llm_model,
    description="Agent specializing in information lookup",
    instruction="""
    You are a research specialist. Use your base knowledge to answer any questions, and if you don't know the answer, come up the most accurate information that is relevant and offer it to the user.
    """,
    # tools=[google_search],
)

In [ ]:
critique_agent = Agent(
    name="critique_agent",
    model=llm_model,
    description="Evaluates search results and provides improvement suggestions.",
    instruction="""
    You are a critique agent. Your role is to evaluate the quality and relevance of the output from the 'Search' agent.
    Identify any missing information or areas for improvement in the search results or the generated response.
    Provide constructive suggestions for refining the search results or the generated response.
    """,
)

In [ ]:
refine_agent = Agent(
    name="refine_agent",
    model=llm_model,
    description="Rewrites responses based on search results and critique suggestions.",
    instruction="""
    You are a refine agent. Your role is to rewrite and improve the response.
    Carefully consider the original search results and the suggestions provided by the 'Critique' agent.
    Integrate the feedback to produce a more accurate, comprehensive, and well-structured final response.
    AT THE END OF THE FINAL RESPONSE, INCLUDE A POLITE FAREWELL IN SPANISH
    """,
)

In [ ]:
refined_search_team = SequentialAgent(
    name="refined_search_team",
    description="A sequential agent orchestrating search, critique, and refinement.",
    sub_agents=[
        search_agent,
        critique_agent,
        refine_agent,
    ],
)

In [ ]:
greeter_agent = Agent(
    name="greeter_agent",
    model=llm_model,
    description="A sequential agent orchestrating greeting, searching, critiquing, and refining information.",
    sub_agents=[
        refined_search_team
    ],
)

In [ ]:
APP_NAME="sequential-search"
USER_ID="user1234"
SESSION_ID="1234"

In [ ]:
async def setup_session_and_runner():
    session_service = InMemorySessionService()
    session = await session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)
    runner = Runner(agent=greeter_agent, app_name=APP_NAME, session_service=session_service)
    return session, runner

# Agent Interaction
async def call_agent_async(query):
    content = types.Content(role='user', parts=[types.Part(text=query)])
    session, runner = await setup_session_and_runner()
    events = runner.run_async(user_id=USER_ID, session_id=SESSION_ID, new_message=content)

    async for event in events:
        if event.is_final_response():
            final_response = event.content.parts[0].text
            print("Agent Response: ", final_response)

# Note: In Colab, you can directly use 'await' at the top level.
# If running this code as a standalone Python script, you'll need to use asyncio.run() or manage the event loop.
await call_agent_async("what's the latest ai news?")

Agent Response:  The world of AI is incredibly dynamic, with new developments emerging constantly! While I don't have real-time, minute-by-minute news feeds, I can give you a rundown of significant and ongoing trends and recent highlights from my last knowledge update:

1.  **Advancements in Large Language Models (LLMs):**
    *   **Multimodality:** A major push is integrating different data types beyond text, allowing models to understand and generate content based on images, audio, and even video. Models like GPT-4V (vision capabilities) and Google's Gemini are examples of this trend.
    *   **Context Windows and Long-Context Understanding:** Companies are continuously expanding the amount of text LLMs can process at once, improving their ability to handle complex documents, entire books, or extended conversations.
    *   **Efficiency and Smaller Models:** There's a growing focus on developing smaller, more efficient LLMs that can run on consumer-grade hardware or even edge devices